# 02. Pipelines and Generation

**Topics covered:** Pipeline API · Text Generation · Sampling Strategies

This notebook builds on [01_loading_models_and_tokenizers.ipynb](https://github.com/S33mi/modern-ai-llm-journey/blob/main/02_huggingface_basics/01_loading_models_and_tokenizers.ipynb).

We will:
1. Use the high-level **Pipeline API** for common NLP tasks
2. Perform **text generation** with `model.generate()`
3. Understand and compare **sampling strategies** (greedy, temperature, top-k, top-p, beam search)
4. Control generation with common parameters (max length, repetition penalty, …)

## 1. Setup & Imports
Run this code once if needed!
```bash
pip install transformers torch
```

In [23]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    set_seed,
)

set_seed(42)
device = 0 if torch.cuda.is_available() else -1   # pipeline uses int device id
torch_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Pipeline device id: {device}")
print(f"Torch device      : {torch_device}")

Pipeline device id: -1
Torch device      : cpu


## 2. The Pipeline API

`pipeline(task, model=...)` is the fastest way to run inference for standard tasks.

It automatically:
- downloads the model & tokenizer
- preprocesses inputs
- runs the model
- post-processes outputs into a convenient Python structure

### Common pipeline tasks

| Task string | Description |
|-------------|-------------|
| `"text-generation"` | Autoregressive language modeling |
| `"text-classification"` | Sentiment, topic, … |
| `"token-classification"` | NER, POS |
| `"question-answering"` | Extractive QA |
| `"summarization"` | Abstractive summary |
| `"translation"` | Machine translation |
| `"fill-mask"` | Masked language modeling |
| `"feature-extraction"` | Raw embeddings |

In [24]:
# Text generation pipeline (uses gpt2 by default if model is omitted)
gen = pipeline(
    "text-generation",
    model="gpt2",  # you can chage this
    device=device,
)

result = gen(
    "Once upon a time in a land far away,",
    max_new_tokens=40,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    truncation=True,
)
print(result[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Once upon a time in a land far away, I would have a chance to have a conversation with someone, to see what they felt would be best. I was not interested in this conversation.

The truth is, there are many ways to


In [25]:
# Sentiment analysis pipeline
classifier = pipeline("sentiment-analysis", device=device)
print(classifier("I love learning about large language models!"))
print(classifier("This tutorial is confusing and poorly written."))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9997605681419373}]
[{'label': 'NEGATIVE', 'score': 0.9998043179512024}]


In [26]:
# Named Entity Recognition
ner = pipeline("token-classification", model="dbmdz/bert-large-cased-finetuned-conll03-english", device=device, aggregation_strategy="simple")
print(ner("Hugging Face is based in New York and was founded by Clément Delangue."))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'ORG', 'score': np.float32(0.9549226), 'word': 'Hugging Face', 'start': 0, 'end': 12}, {'entity_group': 'LOC', 'score': np.float32(0.99880075), 'word': 'New York', 'start': 25, 'end': 33}, {'entity_group': 'PER', 'score': np.float32(0.9947753), 'word': 'Clément Delangue', 'start': 53, 'end': 69}]


## 3. Loading a Model for Manual Generation

For finer control we load the model and tokenizer ourselves (as in the previous notebook).

In [27]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(torch_device)
model.eval()
print(f"Model loaded on {torch_device}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded on cpu


## 4. Text Generation with `model.generate()`

The core method is `model.generate(**inputs, **generation_kwargs)`.

Key arguments you will use constantly:

| Argument | Meaning |
|----------|--------|
| `max_new_tokens` | How many tokens to generate |
| `do_sample` | If `False` → greedy / beam; if `True` → sampling |
| `temperature` | Softmax temperature (higher = more random) |
| `top_k` | Sample only from the k most likely tokens |
| `top_p` | Nucleus sampling – keep the smallest set whose cumulative probability ≥ p |
| `num_beams` | Beam-search width |
| `repetition_penalty` | Penalise tokens that already appeared |
| `no_repeat_ngram_size` | Block repeating n-grams |
| `eos_token_id` / `pad_token_id` | Stopping & padding tokens |

In [28]:
def generate(prompt, **kwargs):
    """Helper that tokenizes, generates and decodes."""
    inputs = tokenizer(prompt, return_tensors="pt").to(torch_device)
    # Sensible defaults
    kwargs.setdefault("max_new_tokens", 40)
    kwargs.setdefault("pad_token_id", tokenizer.eos_token_id)
    with torch.no_grad():
        output_ids = model.generate(**inputs, **kwargs)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


prompt = "The future of artificial intelligence is"
print(generate(prompt, do_sample=False))   # greedy

The future of artificial intelligence is uncertain.

"We're not sure what the future will look like," said Dr. Michael S. Schoenfeld, a professor of computer science at the University of California, Berkeley. "


## 5. Sampling Strategies Explained

### 5.1 Greedy Decoding
Always pick the token with the highest probability. Deterministic but often repetitive.

In [29]:
print("=== Greedy ===")
print(generate(prompt, do_sample=False, max_new_tokens=50))

=== Greedy ===
The future of artificial intelligence is uncertain.

"We're not sure what the future will look like," said Dr. Michael S. Schoenfeld, a professor of computer science at the University of California, Berkeley. "But we're not sure what the future will look


### 5.2 Temperature Sampling

Divide logits by `temperature` before the softmax.

- `temperature → 0` ≈ greedy
- `temperature = 1` = original distribution
- `temperature > 1` = flatter distribution (more creative / random)

In [30]:
for temp in [0.3, 0.7, 1.0, 1.2, 1.9]:    #[0.3, 0.7, 1.2]
    print(f"\n=== temperature ={temp} ===")
    print(generate(prompt, do_sample=True, temperature=temp, max_new_tokens=40))


=== temperature =0.3 ===
The future of artificial intelligence is in the hands of the next generation of computers.

The future of artificial intelligence is in the hands of the next generation of computers.

The future of artificial intelligence is in the hands of

=== temperature =0.7 ===
The future of artificial intelligence is in question.

"If you think about it, if you think about how many of us we have, how many of us we want to learn, how many of us are going to become

=== temperature =1.0 ===
The future of artificial intelligence is uncertain, and it may not have anything to do with that future. Rather, it is going to come down to a number of factors: the increasing complexity of the scientific consensus, a more fundamental shift

=== temperature =1.2 ===
The future of artificial intelligence is a major focus. It's an area of academic research but the real focus seems to have developed well recently and it will continue to develop to this very minute.

We have now come a few

### 5.3 Top-k Sampling

Restrict the sampling distribution to the *k* tokens with the highest probability, then renormalise and sample.

In [31]:
for k in [5, 20, 50]:
    print(f"\n=== top_k = {k} ===")
    print(generate(prompt, do_sample=True, top_k=k, temperature=0.9, max_new_tokens=40))


=== top_k = 5 ===
The future of artificial intelligence is still in its infancy and the technology is still being developed. But it is already a very promising technology, and we will continue to see it evolve and grow in leaps and bounds as more people become aware

=== top_k = 20 ===
The future of artificial intelligence is going to be very interesting.

We have a very complex technology, where a very large number of people are getting it. One thing I would say is that we're already seeing a lot of

=== top_k = 50 ===
The future of artificial intelligence is already at risk, says Dr. Andrew Shafer, a behavioral scientist at the University of Michigan. "The human brain doesn't evolve a lot in the twenty-first century. A major challenge in


### 5.4 Top-p (Nucleus) Sampling

Keep the smallest set of tokens whose cumulative probability mass ≥ `p`, then sample from that set.

This adapts the number of candidates to the confidence of the model.

In [32]:
for p in [0.7, 0.9, 0.95]:
    print(f"\n=== top_p = {p} ===")
    print(generate(prompt, do_sample=True, top_p=p, temperature=0.9, max_new_tokens=40))


=== top_p = 0.7 ===
The future of artificial intelligence is now a question of time. If we can't find a way to improve it, then what's the next step?

As the AI industry moves forward, it is clear that it has many

=== top_p = 0.9 ===
The future of artificial intelligence is an exciting one. At its heart is the question of what might happen to AI when we get to the point where we are able to make intelligent choices, from who will be able to read my emails

=== top_p = 0.95 ===
The future of artificial intelligence is not yet clear. Many scientists believe the future is now in the future of medicine, where the best of these technologies can be tested to produce the desired results. The key to understanding what is to come


### 5.5 Beam Search

Keep the `num_beams` most promising partial hypotheses at each step. More expensive but often higher quality for tasks that need coherence (translation, summarisation).

In [33]:
print("=== Beam search (num_beams = 4) ===\n")
print(generate(
    prompt,
    do_sample=False,
    num_beams=4,
    early_stopping=True,
    max_new_tokens=40,
))

=== Beam search (num_beams = 4) ===

The future of artificial intelligence is in the hands of the next generation of researchers.

"We're going to see a lot of advances in artificial intelligence in the next few years, and we're going to see a lot of


## 6. Controlling Repetition

In [34]:
print("=== No repetition control ===")
print(generate(prompt, do_sample=True, temperature=0.9, max_new_tokens=60))

print("\n=== repetition_penalty = 1.3 ===")
print(generate(prompt, do_sample=True, temperature=0.9, repetition_penalty=1.3, max_new_tokens=60))

print("\n=== no_repeat_ngram_size = 3 ===")
print(generate(prompt, do_sample=True, temperature=0.9, no_repeat_ngram_size=3, max_new_tokens=60))

=== No repetition control ===
The future of artificial intelligence is bright, but there's no saying what's next for artificial intelligence.

The only question is when and how.

=== repetition_penalty = 1.3 ===
The future of artificial intelligence is in question, because its current capabilities are limited to a handful.
"It's not possible for it and our best efforts will continue until the next generation comes along with their own tools [that] can provide what we call "smart vision," so-called by technology scientists (like us). So

=== no_repeat_ngram_size = 3 ===
The future of artificial intelligence is a matter of debate. And indeed, it remains to be seen whether such a future could have come about as predicted by intelligent machines.

A few short paragraphs can be read and understood here:


## 7. Combining Strategies (Practical Recipe)

A widely used combination for open-ended generation:

```python
do_sample=True
temperature=0.7–0.9
top_p=0.9–0.95
top_k=40–50          # optional extra safety
repetition_penalty=1.1–1.2
```

In [35]:
print(generate(
    "In the year 2050, humanity discovered",
    do_sample=True,
    temperature=0.8,
    top_p=0.92,
    top_k=50,
    repetition_penalty=1.15,
    max_new_tokens=80,
))

In the year 2050, humanity discovered that some of its most sensitive organs are being exposed to toxins from fossil fuels and in particular heavy metals like mercury. It has taken many years for these new chemicals to reach humans so they must be avoided if there is hope left on how best we can fight global warming with a less harmful form of medicine such as cancer drugs or vaccines?



## 8. Pipeline vs Manual `generate()` – When to Use Which

| Use case | Prefer |
|----------|--------|
| Quick experiments, demos, standard tasks | **Pipeline** |
| Custom decoding logic, logits inspection, fine control | **Manual `model.generate()`** |
| Batch generation with different parameters per sample | Manual |
| Production micro-services with pre-loaded models | Manual (or a thin wrapper around `generate`) |

In [36]:
# Same generation via pipeline for comparison
pipe_out = gen(
    "In the year 2050, humanity discovered",
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_p=0.92,
    top_k=50,
    repetition_penalty=1.15,
    truncation=True,
)
print(pipe_out[0]["generated_text"])

[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In the year 2050, humanity discovered more than a trillion new species of plants and animals each day. Scientists are currently analyzing this data to learn how they might interact with one another in order to better understand human evolution—and it is not too late for us to take action!


## 9. Summary

| Strategy | Deterministic? | Diversity | Typical use |
|----------|----------------|-----------|-------------|
| **Greedy** | Yes | Low | Debugging, short answers |
| **Beam search** | Yes | Low–medium | Translation, summarisation |
| **Temperature** | No | Controlled by T | Creative writing |
| **Top-k** | No | Medium | General open-ended generation |
| **Top-p (nucleus)** | No | Adaptive | Most modern LLM defaults |

### Quick reference – recommended defaults for open-ended text

```python
model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
)
```

---

**Next notebook:** [`03_embeddings_and_feature_extraction.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/02_huggingface_basics/03_embeddings_and_feature_extraction.ipynb)
Sentence Embeddings · Feature Extraction · Similarity